This notebook create features based on the processed dataset from the previous notebook

In [1]:
import numpy as np
import pandas as pd
import os
os.chdir('../')

from toast_cap.utilities import config

import timeit

In [2]:
start_time = timeit.default_timer()

In [3]:
horizon=60
run_date_str = '20250131'  # queries will pull data up to 1 day prior to run_date_str
s3_output = f's3://toast-datascience-sandbox/PradeepA/Early_Defaults/{run_date_str}' # directory to store the raw data

df = pd.read_parquet(f's3://toast-datascience-sandbox/PradeepA/Early_Defaults/{run_date_str}/full_raw_data.parquet') 
df_orders = pd.read_parquet(f's3://toast-datascience-sandbox/PradeepA/Early_Defaults/{run_date_str}/orders.parquet') 
df_loans = pd.read_parquet(f's3://toast-datascience-sandbox/PradeepA/Early_Defaults/{run_date_str}/loans.parquet') 
df_middesk = pd.read_parquet(f's3://toast-datascience-sandbox/PradeepA/Early_Defaults/{run_date_str}/middesk.parquet') 
df_payroll_bounce = pd.read_parquet(f's3://toast-datascience-sandbox/PradeepA/Early_Defaults/{run_date_str}/payroll_bounce.parquet')
df_rid_guid_bridge = pd.read_parquet(f's3://toast-datascience-sandbox/PradeepA/Early_Defaults/{run_date_str}/rx_guid_rid_bridge.parquet') 


# df_defaults = pd.read_parquet(f's3://toast-datascience-sandbox/PradeepA/Early_Defaults/{run_date_str}/tc_default.parquet') # read 'full_raw_data.parquet' generated from the previous notebook

# you can add some filters to reduce the dataset you're going to work with
df = df.loc[(~df[f'label_{horizon}'].isna()) & (df['dt'].dt.year>=2018)]
df = df.sort_values(by=['rid', 'dt'])

In [4]:
df_rid_guid_bridge = pd.read_parquet(f's3://toast-datascience-sandbox/PradeepA/Early_Defaults/{run_date_str}/rx_guid_rid_bridge.parquet')
df_rid_guid_bridge.head()

,rid,restaurant_guid
0,162379000000000000,01d1c001-6c46-4061-b72a-cb5a4799bf55
1,47445000000000000,4403c864-4c4f-4dfc-897f-31e865819768
2,28468000000000000,6ffde169-53a7-43a2-a94b-e6d40f9f98e4
3,168613000000000000,e19736eb-48f5-4368-a0b2-366dd98f10af
4,197906000000000000,6c92cffa-40bf-4404-930e-b613d19f9d8e


In [5]:
df_loans['rid'] = df_loans['rid'].astype('int64')

In [6]:
df_loans.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 54785 entries, 0 to 54784
Data columns (total 25 columns):
 #   Column                                  Non-Null Count  Dtype         
---  ------                                  --------------  -----         
 0   loan_id                                 54785 non-null  object        
 1   rid                                     54785 non-null  int64         
 2   entity_identifier                       54785 non-null  object        
 3   funding_partner                         54785 non-null  object        
 4   customer_id                             54687 non-null  object        
 5   customer_name                           54687 non-null  object        
 6   created_date                            54762 non-null  datetime64[ns]
 7   is_pre_90                               45407 non-null  object        
 8   term_in_days                            54785 non-null  int64         
 9   ineligible_date                         5704 non-n

In [7]:
df_loans_rids = df_loans.drop_duplicates(subset=['rid'])

In [8]:
df = pd.merge(df,df_loans_rids[['rid']],on = ['rid'], how='inner')

In [9]:
df.head(100)

,rid,dt,totalRevenue,totalCreditCardRevenue,tx_hours,noprocessing,label_60,state,first_order_date,parent_market_segment,...,gpv_date_min,first_loan_date,first_loan_date_90d,first_loan_date_270d,first_loan_date_360d,yearmon,live_online_ordering_module_count,live_saas_mrr,live_saas_module_count,live_gift_card_module_count
0,45000000000000,2020-04-01,30.00,30.00,1.0,NaN,0.0,MA,2013-10-18,SMB,...,2020-04-01,2023-08-08,2023-08-08,NaT,NaT,202004,0.0,150.0,4.0,1.0
1,45000000000000,2020-04-02,0.00,0.00,0.0,NaN,0.0,MA,2013-10-18,SMB,...,2020-04-01,2023-08-08,2023-08-08,NaT,NaT,202004,0.0,150.0,4.0,1.0
2,45000000000000,2020-04-03,0.00,0.00,0.0,NaN,0.0,MA,2013-10-18,SMB,...,2020-04-01,2023-08-08,2023-08-08,NaT,NaT,202004,0.0,150.0,4.0,1.0
3,45000000000000,2020-04-04,0.00,0.00,0.0,NaN,0.0,MA,2013-10-18,SMB,...,2020-04-01,2023-08-08,2023-08-08,NaT,NaT,202004,0.0,150.0,4.0,1.0
4,45000000000000,2020-04-05,0.00,0.00,0.0,NaN,0.0,MA,2013-10-18,SMB,...,2020-04-01,2023-08-08,2023-08-08,NaT,NaT,202004,0.0,150.0,4.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,45000000000000,2020-07-05,2750.40,1677.59,9.0,0.0,0.0,MA,2013-10-18,SMB,...,2020-04-01,2023-08-08,2023-08-08,NaT,NaT,202007,0.0,150.0,4.0,1.0
96,45000000000000,2020-07-06,1603.39,1074.00,6.0,0.0,0.0,MA,2013-10-18,SMB,...,2020-04-01,2023-08-08,2023-08-08,NaT,NaT,202007,0.0,150.0,4.0,1.0
97,45000000000000,2020-07-07,2348.29,1307.48,6.0,0.0,0.0,MA,2013-10-18,SMB,...,2020-04-01,2023-08-08,2023-08-08,NaT,NaT,202007,0.0,150.0,4.0,1.0
98,45000000000000,2020-07-08,2584.89,1595.72,9.0,0.0,0.0,MA,2013-10-18,SMB,...,2020-04-01,2023-08-08,2023-08-08,NaT,NaT,202007,0.0,150.0,4.0,1.0


In [10]:
df[df['rid'] == 45000000000000].to_csv('45000000000000.csv')

In [11]:
# df['yyyymmdd'] = df['dt'].dt.strftime('%Y%m%d').astype(int)
df['yyyymmdd'] = pd.to_datetime(df.dt,format="%Y%m%d")

df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16113058 entries, 0 to 16113057
Data columns (total 24 columns):
 #   Column                             Dtype         
---  ------                             -----         
 0   rid                                int64         
 1   dt                                 datetime64[ns]
 2   totalRevenue                       float64       
 3   totalCreditCardRevenue             float64       
 4   tx_hours                           float64       
 5   noprocessing                       float64       
 6   label_60                           float64       
 7   state                              object        
 8   first_order_date                   datetime64[ns]
 9   parent_market_segment              object        
 10  restaurant_type                    object        
 11  account_restaurant_category        object        
 12  tx_date_min                        datetime64[ns]
 13  gpv_date_min                       datetime64[ns]
 14  

In [12]:
df = pd.merge(df,df_orders,
                   left_on = [config.GROUP,'yyyymmdd'],
                   right_on = [config.GROUP,'yyyymmdd'],
                   how='left'
                  )

df.drop(columns=['yyyymmdd'],inplace=True)
    
## impute null after left join
print("Impute null values after left join")

df['amount_sum'].fillna(0,inplace=True)
df['total_orders'].fillna(0,inplace=True)

Impute null values after left join


In [13]:
## Config file
# Feature upper & lower bounds
GMV_90D_Percent_Change_upper_clip = 500
GMV_90D_Percent_Change_lower_clip = -100
GMV_YoY_90D_Percent_Change_upper_clip = 500
GMV_YoY_90D_Percent_Change_lower_clip = -100
amount_sum_upper_clip = 28266.589844
total_orders_upper_clip = 3000

## features from orders data 
print("Calculate orders feature")
df = df.sort_values(by=[config.GROUP,'dt'], ascending=True, na_position='first')
df['amount_sum'] = df.amount_sum.clip(upper=amount_sum_upper_clip)
df['total_orders'] = df.total_orders.clip(upper=total_orders_upper_clip)

df['total_no_of_orders_30d'] = df.set_index('dt').groupby(config.GROUP)\
['total_orders'].rolling('30D').sum().values

df['median_no_of_orders_180d'] = df.set_index('dt').groupby(config.GROUP)['total_orders']\
.rolling('180D').median().values

df['std_orders_180d'] = df.set_index('dt')\
.groupby(config.GROUP)['total_orders'].rolling('180D').std().values

df['gmv_sum_30d'] = df.set_index('dt').groupby(config.GROUP)['amount_sum']\
.rolling('30D').sum().values

print("## growth feature using avereage df amount ##") 

df['total_size_orders_90'] = \
df.set_index('dt').groupby(config.GROUP)\
['amount_sum'].rolling('90D',min_periods=30).mean().values

df['GMV_90D_Percent_Change'] = df.groupby([config.GROUP])\
['total_size_orders_90'].pct_change(90)*100

df['GMV_YoY_90D_Percent_Change'] = df.groupby([config.GROUP])\
['total_size_orders_90'].pct_change(365)*100

df['GMV_90D_Percent_Change'] = df.GMV_90D_Percent_Change.clip(lower=GMV_90D_Percent_Change_lower_clip,
                                                                    upper=GMV_90D_Percent_Change_upper_clip)
df['GMV_YoY_90D_Percent_Change'] = df.GMV_YoY_90D_Percent_Change.clip(lower=GMV_YoY_90D_Percent_Change_lower_clip,
                                                                            upper=GMV_YoY_90D_Percent_Change_upper_clip)

Calculate orders feature
## growth feature using avereage df amount ##


In [14]:
# time on Toast
df['days_with_toast'] = ((df['dt'] - pd.to_datetime(df['first_order_date'])).dt.days).clip(lower=0)
df['months_with_toast'] = ((df.dt - df.first_order_date)/np.timedelta64(1, 'M'))
df['months_with_toast'] = df['months_with_toast'].astype(int)

In [15]:
def encode_account_restaurant_category(x):
    category_map = {
        'Brewery': 1,
        'FSR': 2,
        'FSR - Bar': 3,
        'FSR - Casual': 4,
        'FSR - Casual Dining': 5,
        'FSR - Diner': 6,
        'FSR - Fine Dining': 7,
        'FSR - Hotel': 8,
        'FSR - Other': 9,
        'FSR - Pizzeria': 10,
        'Fast Casual': 11,
        'Ghost Kitchen': 12,
        'Hotel - Full Service': 13,
        'Not a Restaurant': 14,
        'Other': 15,
        'Pizzeria': 16,
        'QSR': 17,
        'QSR - Bakery/Desserts': 18,
        'QSR - Cafe / Bakery': 19,
        'QSR - Cafe/Deli/Lunch': 20,
        'QSR - Concessions and Stadiums': 21,
        'QSR - Fast Casual': 22,
        'QSR - Fast Food / Drivethrough': 23,
        'QSR - Food Truck': 24,
        'QSR - Institutional Dining': 25,
        'QSR - Nightclub': 26,
        'QSR - Other': 27,
        'QSR - Pizzeria': 28,
        'Restaurant - Full Service': 29,
        'Restaurant - Office': 30,
        'Retail - Bottle Shop/Liquor Store': 31,
        'Retail - Convenience Store': 32,
        'Retail - Grocery/Market': 33,
        'Retail - Other': 34,
        '': 99
    }
    
    return category_map.get(x, 98)

df['account_restaurant_category'] = df['account_restaurant_category'].apply(encode_account_restaurant_category)

def encode_parent_market_segment(x):
    segment_map = {
        'Enterprise': 0,
        'Mid Market': 1,
        'Regional Mid Market': 2,
        'SMB': 3,
        '': 99
    }
    
    return segment_map.get(x, 98)

df['parent_market_segment'] = df['parent_market_segment'].apply(encode_parent_market_segment)


In [16]:
# modules
df['has_oo_mod'] = np.where(
    df['live_online_ordering_module_count'] > 0, 1, 0
)
df['has_gc_mod'] = np.where(
    df['live_gift_card_module_count'] > 0, 1, 0
)

In [17]:
df['gpv_mean_365d'] = df.set_index('dt').groupby(config.GROUP).rolling('365d')['totalCreditCardRevenue'].mean().values
df['gpv_mean_180d'] = df.set_index('dt').groupby(config.GROUP).rolling('180d')['totalCreditCardRevenue'].mean().values


df['gpv_mean_90d'] = df.set_index('dt').groupby(config.GROUP).rolling('90d')['totalCreditCardRevenue'].mean().values

## volatility
df['gpv_cv_180d'] = df.set_index('dt').groupby(config.GROUP).rolling('180d')['totalCreditCardRevenue'].std().values / df['gpv_mean_180d'].values
df['gpv_cv_90d'] = df.set_index('dt').groupby(config.GROUP).rolling('90d')['totalCreditCardRevenue'].std().values / df['gpv_mean_90d'].values

df['log_gpv'] = np.where(df['totalCreditCardRevenue']>0, np.log(df['totalCreditCardRevenue']+1), 0)
# df['log_gpv_std_180d'] = df.set_index('dt').groupby(config.GROUP).rolling('180d')['log_gpv'].std().values
df['log_gpv_std_90d'] = df.set_index('dt').groupby(config.GROUP).rolling('90d')['log_gpv'].std().values
df = df.drop(columns='log_gpv')

df['gpv_mean_28d'] = df.set_index('dt').groupby(config.GROUP).rolling('28d')['totalCreditCardRevenue'].mean().values

df['gpv_median_28d'] = df.set_index('dt').groupby(config.GROUP).rolling('28d')['totalCreditCardRevenue'].median().values

df['gpv_median_84d'] = df.set_index('dt').groupby(config.GROUP).rolling('84d')['totalCreditCardRevenue'].median().values

df['gpv_median_180d'] = df.set_index('dt').groupby(config.GROUP).rolling('180d')['totalCreditCardRevenue'].median().values

df['gpv_median_28d_median_180d_diff'] = df['gpv_median_28d'] - df['gpv_median_180d']

df['gpv_median_28d_median_84d_diff'] = df['gpv_median_28d'] - df['gpv_median_84d']

df['gpv_median_28d_mean_90d_diff'] = df['gpv_median_28d'] - df['gpv_mean_90d']

df['gpv_median_28d_28ddiff']= df.groupby(config.GROUP)['gpv_median_28d'].diff(28).fillna(0).values
df['gpv_median_28d_84ddiff']= df.groupby(config.GROUP)['gpv_median_28d'].diff(84).fillna(0).values

/tmp/ipykernel_38404/2391422381.py:8: RuntimeWarning: invalid value encountered in divide
  df['gpv_cv_180d'] = df.set_index('dt').groupby(config.GROUP).rolling('180d')['totalCreditCardRevenue'].std().values / df['gpv_mean_180d'].values
/tmp/ipykernel_38404/2391422381.py:9: RuntimeWarning: invalid value encountered in divide
  df['gpv_cv_90d'] = df.set_index('dt').groupby(config.GROUP).rolling('90d')['totalCreditCardRevenue'].std().values / df['gpv_mean_90d'].values


In [18]:
# GPV per hour trend
df['gpv_per_hour'] = np.where(df['tx_hours'] == 0, 0,df['totalCreditCardRevenue'] / df['tx_hours'])

df['gpv_per_hour_median_28d'] = df.set_index('dt').groupby(config.GROUP).rolling('28d')['gpv_per_hour'].median().values

df['gpv_per_hour_median_28d_28ddiff']= df.groupby(config.GROUP)['gpv_per_hour_median_28d'].diff(28).fillna(0).values

In [19]:
# days with no GPV
df['flag_no_gpv'] = np.where(df['totalCreditCardRevenue']<=0, 1, 0)

df['days_no_gpv_90d'] = df.set_index('dt').groupby(config.GROUP)\
    .rolling('90d')['flag_no_gpv'].sum().values
df['days_no_gpv_28d'] = df.set_index('dt').groupby(config.GROUP)\
    .rolling('28d')['flag_no_gpv'].sum().values

df['days_no_gpv_28d_28ddiff'] = df.groupby(config.GROUP)['days_no_gpv_28d'].diff(28).fillna(0).values

df = df.drop(columns='flag_no_gpv')

In [20]:
# transaction hours
df['tx_hours_mean_14d'] = df.set_index('dt').groupby(
    config.GROUP
).rolling('14d')['tx_hours'].mean().values
df['tx_hours_median_14d'] = df.set_index('dt').groupby(
    config.GROUP
).rolling('14d')['tx_hours'].median().values

df['tx_hours_median_28d'] = df.set_index('dt').groupby(
    config.GROUP
).rolling('28d')['tx_hours'].median().values

df['tx_hours_median_28d_28ddiff'] = df.groupby(config.GROUP)['tx_hours_median_28d'].diff(28).fillna(0).values

In [21]:
# modules
df['has_oo_mod'] = np.where(
    df['live_online_ordering_module_count'] > 0, 1, 0
)
df['has_gc_mod'] = np.where(
    df['live_gift_card_module_count'] > 0, 1, 0
)

In [22]:
# 30-day no processing history
df['noprocessing_last_90d'] = df.set_index('dt').groupby(
    config.GROUP
).rolling('60d')['noprocessing'].max().values  #rolling 150d because 150+30 = 180

# 30-day no processing history
df['noprocessing_last_180d'] = df.set_index('dt').groupby(
    config.GROUP
).rolling('150d')['noprocessing'].max().values  #rolling 150d because 150+30 = 180

In [23]:
end_time = timeit.default_timer()
print(f"The time taken to run the code is {end_time - start_time} seconds.")

The time taken to run the code is 698.1768924547359 seconds.


In [24]:
start_time = timeit.default_timer()

# flag - check if within the last 7 days, there are at least 3 non consecutive days with greater than $50 in credit card sales.
def check_flag(amounts):
    return int((amounts >= 50).sum() >= 3)

# Add a column for the rolling window flag
df['flag_ge50_3d_over_7d'] = df.groupby(config.GROUP)['totalCreditCardRevenue'].rolling(window=7, min_periods=1).apply(check_flag).reset_index(level=0, drop=True).astype(int)

# df['flag_ge50_3d_over_7d'] = df.set_index('dt').groupby(config.GROUP)['totalCreditCardRevenue'].rolling(window=7, min_periods=1).apply(check_flag).reset_index(level=0, drop=True).astype(int)

end_time = timeit.default_timer()
print(f"The time taken to run the code is {end_time - start_time} seconds.")

The time taken to run the code is 2403.290040205233 seconds.


In [25]:
df['flag_ge50_3d_over_7d'].value_counts()

1    15450862
0      662196
Name: flag_ge50_3d_over_7d, dtype: int64

In [26]:
# 30-day no processing history
df['noprocessing_last_180d'] = df.set_index('dt').groupby(
    config.GROUP
).rolling('150d')['noprocessing'].max().values  #rolling 150d because 150+30 = 180

# 30-day no processing history
df['noprocessing_last_180d'] = df.set_index('dt').groupby(
    config.GROUP
).rolling('150d')['noprocessing'].max().values  #rolling 150d because 150+30 = 180

In [27]:
df.head()

,rid,dt,totalRevenue,totalCreditCardRevenue,tx_hours,noprocessing,label_60,state,first_order_date,parent_market_segment,...,days_no_gpv_90d,days_no_gpv_28d,days_no_gpv_28d_28ddiff,tx_hours_mean_14d,tx_hours_median_14d,tx_hours_median_28d,tx_hours_median_28d_28ddiff,noprocessing_last_90d,noprocessing_last_180d,flag_ge50_3d_over_7d
0,45000000000000,2020-04-01,30.0,30.0,1.0,NaN,0.0,MA,2013-10-18,3,...,0.0,0.0,0.0,1.000000,1.0,1.0,0.0,NaN,NaN,0
1,45000000000000,2020-04-02,0.0,0.0,0.0,NaN,0.0,MA,2013-10-18,3,...,1.0,1.0,0.0,0.500000,0.5,0.5,0.0,NaN,NaN,0
2,45000000000000,2020-04-03,0.0,0.0,0.0,NaN,0.0,MA,2013-10-18,3,...,2.0,2.0,0.0,0.333333,0.0,0.0,0.0,NaN,NaN,0
3,45000000000000,2020-04-04,0.0,0.0,0.0,NaN,0.0,MA,2013-10-18,3,...,3.0,3.0,0.0,0.250000,0.0,0.0,0.0,NaN,NaN,0
4,45000000000000,2020-04-05,0.0,0.0,0.0,NaN,0.0,MA,2013-10-18,3,...,4.0,4.0,0.0,0.200000,0.0,0.0,0.0,NaN,NaN,0


In [28]:
# Calculate missing stats for each variable
missing_stats = pd.DataFrame({
    'Total Count': df.count(),  # Non-missing values count
    'Missing Count': df.isnull().sum(),  # Missing values count
    '% Missing': df.isnull().mean() * 100  # Percentage of missing values
})

# Sort by % Missing in descending order
missing_stats = missing_stats.sort_values(by='% Missing', ascending=False)
print(missing_stats)

                             Total Count  Missing Count  % Missing
first_loan_date_90d              6439210        9673848  60.037319
GMV_YoY_90D_Percent_Change       9397529        6715529  41.677557
first_loan_date_270d             9715242        6397816  39.705784
first_loan_date_360d             9778372        6334686  39.313990
GMV_90D_Percent_Change          13930465        2182593  13.545492
...                                  ...            ...        ...
tx_hours_mean_14d               16113058              0   0.000000
tx_hours_median_28d             16113058              0   0.000000
tx_hours_median_14d             16113058              0   0.000000
tx_hours_median_28d_28ddiff     16113058              0   0.000000
flag_ge50_3d_over_7d            16113058              0   0.000000

[64 rows x 3 columns]


In [29]:
df['yearmon'].head()

0    202004
1    202004
2    202004
3    202004
4    202004
Name: yearmon, dtype: int64

In [30]:
df = df[df['yearmon']>=202201]
df['yearmon'].value_counts(dropna=False, ascending=True)
yearmon_counts = df['yearmon'].value_counts(dropna=False).sort_index()

print(yearmon_counts)

202201    236488
202202    221604
202203    254591
202204    255101
202205    273552
202206    274669
202207    294874
202208    305582
202209    304724
202210    324442
202211    323244
202212    343284
202301    351645
202302    324927
202303    367672
202304    366238
202305    389085
202306    385421
202307    405461
202308    411769
202309    404944
202310    424379
202311    415606
202312    433717
202401    435688
202402    409284
202403    440132
202404    428571
202405    445800
202406    433985
202407    448423
202408    447213
202409    431041
202410    442344
202411    422677
202412     13976
Name: yearmon, dtype: int64


In [31]:
# Calculate missing stats for each variable
missing_stats = pd.DataFrame({
    'Total Count': df.count(),  # Non-missing values count
    'Missing Count': df.isnull().sum(),  # Missing values count
    '% Missing': df.isnull().mean() * 100  # Percentage of missing values
})

# Sort by % Missing in descending order
missing_stats = missing_stats.sort_values(by='% Missing', ascending=False)
print(missing_stats)

                             Total Count  Missing Count  % Missing
first_loan_date_90d              5060008        7832145  60.751257
first_loan_date_270d             7527269        5364884  41.613561
first_loan_date_360d             7959164        4932989  38.263500
GMV_YoY_90D_Percent_Change       8514759        4377394  33.953941
GMV_90D_Percent_Change          11553836        1338317  10.380865
...                                  ...            ...        ...
tx_hours_mean_14d               12892153              0   0.000000
tx_hours_median_28d             12892153              0   0.000000
tx_hours_median_14d             12892153              0   0.000000
tx_hours_median_28d_28ddiff     12892153              0   0.000000
flag_ge50_3d_over_7d            12892153              0   0.000000

[64 rows x 3 columns]


In [32]:
df_middesk.columns = df_middesk.columns.str.lower().str.replace(" ", "_")

In [33]:
df_middesk['liens']

0                                                       []
1                                                       []
2        [\n  {\n    "amountUSD": 0,\n    "collateral":...
3        [\n  {\n    "amountUSD": 0,\n    "collateral":...
4                                                       []
                               ...                        
76382                                                 None
76383                                                 None
76384                                                 None
76385                                                 None
76386                                                 None
Name: liens, Length: 76387, dtype: object

# Create Payroll Bounce features

In [34]:
def encode_reason_for_bounce(x):
    bounce_map = {
        'Insufficient Funds': 0,
        'Unauthorized Corporate Debit': 1,
        'Account Frozen': 2,
        'Account Closed': 3,
        'Split Payment': 4,
        'Payment Stopped': 5,
        'No Account/Cannot Locate': 6,
        'Invalid ACH Routing Number': 7,
        'Invalid Account Number': 8,
        'Non Transaction Acct': 9,
        'Uncollected Funds': 10,
        'T/R Check Digit Error': 11,
        'Receiver Refuses Payment': 12,
        'Authorization Revoked': 13,
        'Corp Entry To Consumer': 14,
        'Beneficiary Deceased': 15,
        'Vendor Payment': 16,
        'Limited Participation DFI': 17,
        'Permissible Return': 18,
        'File Rec Edit Criteria': 19,
        'Orig Unknown/Not Auth': 20,
        'Other': 98
    }

    # Normalize input (strip spaces, convert to title case, remove newlines)
    x = x.strip().replace("\n", "").title()

    # Handle common variations
    standardize_map = {
        'No Acct/Cannot Locate': 'No Account/Cannot Locate',
        'No Account / Cannot Locate': 'No Account/Cannot Locate',
        'No AccountCannot Locate': 'No Account/Cannot Locate',
        'No Account Cannot Locate': 'No Account/Cannot Locate',
        'Non Transaction Account': 'Non Transaction Acct',
        'Non Transaction Acc': 'Non Transaction Acct',
        'Non Transaction Acct\n': 'Non Transaction Acct',
        'Uncollected Funds\n': 'Uncollected Funds',
        'T/R Check Digit Error\n': 'T/R Check Digit Error',
        'Corp Entry To Consumer\n': 'Corp Entry To Consumer',
        'Unauthorized Corporate Debit': 'Unauthorized Corporate Debit',
        'Unauth Corp Debit': 'Unauthorized Corporate Debit',
        'Split Paymetn': 'Split Payment',
        'Split Paymentj': 'Split Payment',
        'Splite Payment': 'Split Payment',
        'Split payment': 'Split Payment',
        'split payment': 'Split Payment'
    }

    # Apply standardization if exists, else return the mapped value or "Other"
    return bounce_map.get(standardize_map.get(x, x), 98)

df_payroll_bounce['encoded_reason_for_bounce'] = df_payroll_bounce['reason_for_bounce'].apply(encode_reason_for_bounce)
print(df_payroll_bounce[['reason_for_bounce', 'encoded_reason_for_bounce']].drop_duplicates())

                  reason_for_bounce  encoded_reason_for_bounce
0                Insufficient Funds                          0
139                   Split Payment                          4
209                  Account Frozen                          2
257            NON TRANSACTION ACCT                          9
260            Non Transaction Acct                          9
272    Unauthorized Corporate Debit                          1
308          NON TRANSACTION ACCT\n                          9
337        No Account/Cannot Locate                          6
1308                Payment Stopped                          5
1479          T/R Check Digit Error                         11
1558         Invalid Account Number                          8
1561              Uncollected Funds                         10
1620     Invalid ACH Routing Number                         98
1968                 Account Closed                          3
2162              UNCOLLECTED FUNDS                    

In [35]:
df_payroll_bounce = pd.merge(df_payroll_bounce,df_rid_guid_bridge,left_on='restaurant_guid',right_on='restaurant_guid', how='inner')
df_payroll_bounce.shape

(31305, 37)

In [36]:
df = pd.merge(df,df_rid_guid_bridge,left_on='rid',right_on='rid', how='inner')

In [37]:
unique_rids = df['rid'].unique()
print(f"Number of unique rids: {len(unique_rids)}")

Number of unique rids: 18231


In [38]:
df_inner = df.merge(df_payroll_bounce, on='rid', how='inner')
unique_rids = df_inner['rid'].unique()
print(f"Number of unique rids: {len(unique_rids)}")

Number of unique rids: 1848


In [39]:

# Ensure datetime format
df['dt'] = pd.to_datetime(df['dt'])
df_payroll_bounce['payroll_bounce_date'] = pd.to_datetime(df_payroll_bounce['payroll_bounce_date'])

# Define lookback windows in days
lookback_windows = {
    '1m': 30,
    '2m': 60,
    '3m': 90,
    '6m': 180
}

# Step 1: Merge df with df_payroll_bounce (keeping payroll data within 6 months of dt)
df_merged = df.merge(df_payroll_bounce, on='rid', how='left')

# Step 2: Filter payroll bounces to only those within 6 months of dt
df_merged = df_merged[
    (df_merged['payroll_bounce_date'] > df_merged['dt'] - pd.Timedelta(days=180)) &
    (df_merged['payroll_bounce_date'] <= df_merged['dt'])
]

# Step 3: Compute payroll bounce counts, fee sums, bounced amount sums, and latest bounce reason
for window_name, window_days in lookback_windows.items():
    window_start = df_merged['dt'] - pd.Timedelta(days=window_days)

    # Filter payroll bounces within the window
    df_window = df_merged[
        (df_merged['payroll_bounce_date'] > window_start) &
        (df_merged['payroll_bounce_date'] <= df_merged['dt'])
    ]

    # Group by `rid` and `dt`, then compute:
    # - Count of payroll bounces
    # - Sum of `fee`
    # - Sum of `bounced_amount`
    bounce_counts = df_window.groupby(['rid', 'dt']).size().reset_index(name=f'payroll_bounce_count_{window_name}')
    fee_sums = df_window.groupby(['rid', 'dt'])['fee'].sum().reset_index(name=f'payroll_bounce_fee_sum_{window_name}')
    bounced_amount_sums = df_window.groupby(['rid', 'dt'])['bounced_amount'].sum().reset_index(name=f'payroll_bounce_amount_sum_{window_name}')

    # Capture latest bounce reason within the window
    latest_bounce_reason = df_window.loc[df_window.groupby(['rid', 'dt'])['payroll_bounce_date'].idxmax(), ['rid', 'dt', 'encoded_reason_for_bounce']]
    latest_bounce_reason.rename(columns={'encoded_reason_for_bounce': f'latest_bounce_reason_{window_name}'}, inplace=True)

    # Merge results back into df
    # df = df.merge(bounce_counts, on=['rid', 'dt'], how='left').fillna(0)
    # df = df.merge(fee_sums, on=['rid', 'dt'], how='left').fillna(0)
    # df = df.merge(bounced_amount_sums, on=['rid', 'dt'], how='left').fillna(0)
    df = df.merge(bounce_counts, on=['rid', 'dt'], how='left')
    df = df.merge(fee_sums, on=['rid', 'dt'], how='left')
    df = df.merge(bounced_amount_sums, on=['rid', 'dt'], how='left')
    df = df.merge(latest_bounce_reason, on=['rid', 'dt'], how='left')

# Fill missing values for latest bounce reason with -1 (or another placeholder if needed)
for window_name in lookback_windows.keys():
    df[f'latest_bounce_reason_{window_name}'] = df[f'latest_bounce_reason_{window_name}'].fillna(-1).astype(int)

In [40]:
df.columns

Index(['rid', 'dt', 'totalRevenue', 'totalCreditCardRevenue', 'tx_hours',
       'noprocessing', 'label_60', 'state', 'first_order_date',
       'parent_market_segment', 'restaurant_type',
       'account_restaurant_category', 'tx_date_min', 'gpv_date_min',
       'first_loan_date', 'first_loan_date_90d', 'first_loan_date_270d',
       'first_loan_date_360d', 'yearmon', 'live_online_ordering_module_count',
       'live_saas_mrr', 'live_saas_module_count',
       'live_gift_card_module_count', 'amount_sum', 'total_orders',
       'total_no_of_orders_30d', 'median_no_of_orders_180d', 'std_orders_180d',
       'gmv_sum_30d', 'total_size_orders_90', 'GMV_90D_Percent_Change',
       'GMV_YoY_90D_Percent_Change', 'days_with_toast', 'months_with_toast',
       'has_oo_mod', 'has_gc_mod', 'gpv_mean_365d', 'gpv_mean_180d',
       'gpv_mean_90d', 'gpv_cv_180d', 'gpv_cv_90d', 'log_gpv_std_90d',
       'gpv_mean_28d', 'gpv_median_28d', 'gpv_median_84d', 'gpv_median_180d',
       'gpv_median_28d_me

In [41]:
df['latest_bounce_reason_1m'].value_counts(dropna=False)

-1     12730122
 0       146360
 5         3074
 1         2585
 3         2321
 2         1836
 10        1791
 6         1594
 9          833
 4          509
 98         484
 8          397
 11         157
 14          60
 18          30
Name: latest_bounce_reason_1m, dtype: int64

In [42]:
# # Get the latest payroll bounce reason
# df_merged = df.merge(df_payroll_bounce[['rid', 'payroll_bounce_date', 'encoded_reason_for_bounce']], on='rid', how='left')

# # Function to get latest bounce reason within a lookback window
# def get_latest_bounce_reason(group, window_days):
#     window_start = group['dt'] - pd.Timedelta(days=window_days)
#     window_bounces = group[
#         (group['payroll_bounce_date'] >= window_start) & (group['payroll_bounce_date'] <= group['dt'])
#     ]
    
#     if not window_bounces.empty:
#         latest_bounce = window_bounces.loc[window_bounces['payroll_bounce_date'].idxmax()]
#         return latest_bounce['encoded_reason_for_bounce']
#     return None  # No bounce in the window

# # Apply the function to capture the latest bounce reason in different lookback windows
# for window in [30]:  # 1M, 2M, 3M, 6M
#     df_merged[f'latest_bounce_reason_{window}d'] = df_merged.groupby('rid').apply(
#         lambda group: get_latest_bounce_reason(group, window)
#     ).reset_index(level=0, drop=True)

# # Final dataframe with latest bounce reasons
# df = df_merged

In [43]:
df['first_loan_date'] = pd.to_datetime(df['first_loan_date'], errors='coerce')
df['first_loan_date_90d'] = pd.to_datetime(df['first_loan_date_90d'], errors='coerce')
df['first_loan_date_270d'] = pd.to_datetime(df['first_loan_date_270d'], errors='coerce')
df['first_loan_date_360d'] = pd.to_datetime(df['first_loan_date_360d'], errors='coerce')

# Export Data

In [44]:
df.to_parquet(os.path.join(s3_output, 'processed_train.parquet'))